# UAV wildlife training workflow — bring your own data

This notebook documents the final Thermal and RGB YOLOv8n training settings. **Supply your own YOLO-format data before training.** The original third-party training/validation dataset is not distributed here, and project-trained checkpoints are not currently included.

You can review the [interactive report](https://bodhi584.github.io/multimodal-uav-wildlife-detection-yolov8/) and [saved experiment evidence](https://github.com/bodhi584/multimodal-uav-wildlife-detection-yolov8/tree/main/assets) without a GPU or dataset. Running all cells with the defaults below prints setup guidance and skips training.

The report describes the original full data-engineering pipeline. This public notebook starts with already prepared YOLO data; it does not perform the original video extraction, annotation, or dataset construction. Training on different data will produce different metrics. Exact metric reproduction requires the corresponding evaluation split, labels, checkpoints, and software settings.

In [ ]:
%pip install -q ultralytics

from collections import Counter
from pathlib import Path
import torch
import ultralytics
import yaml
from ultralytics import YOLO

print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("To train, select Runtime > Change runtime type > GPU, then rerun this cell.")
print("Installed versions are recorded above; the original experiment environment is not pinned here.")

## 1. Configure and inspect your own datasets

Upload/extract your prepared data under `/content/datasets`, or set `USE_GOOGLE_DRIVE = True` to mount your own Drive and change `DATA_ROOT` to its actual folder (for example `/content/drive/MyDrive/datasets`). Colab local files are temporary and are lost when the runtime is deleted.

Default layout: `YOLO_Thermal/{data.yaml,images/{train,val},labels/{train,val}}` and the same for `YOLO_RGB`. Each YAML must point to the actual image directories. Set `MODALITIES` to just `("thermal",)` or `("rgb",)` if you only have one stream. The original class order is Human, Vehicle, Deer, Hare, Dog, Duck; class names below are read from your YAML.

Empty `.txt` files are valid negative frames. This basic inspection checks the example folder layout and image/label pairing. Training stays off until you explicitly set `RUN_TRAINING = True`.

In [ ]:
USE_GOOGLE_DRIVE = False
DATA_ROOT = Path("/content/datasets")  # Change to your actual dataset folder.
MODALITIES = ("thermal", "rgb")
RUN_TRAINING = False  # Enable only after data setup and inspection.

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

FINAL_RUNS = {
    "thermal": {"data": DATA_ROOT / "YOLO_Thermal/data.yaml", "imgsz": 640},
    "rgb": {"data": DATA_ROOT / "YOLO_RGB/data.yaml", "imgsz": 1280},
}
if not MODALITIES or any(m not in FINAL_RUNS for m in MODALITIES):
    raise ValueError("MODALITIES must select thermal, rgb, or both.")

ready_modalities = []
image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
for modality in MODALITIES:
    config = FINAL_RUNS[modality]
    root = config["data"].parent
    if not config["data"].is_file():
        print(f"{modality}: dataset not configured. Missing {config['data']}")
        print("Supply your own data and update DATA_ROOT; the original dataset is not included.")
        continue
    dataset_config = yaml.safe_load(config["data"].read_text()) or {}
    names = dataset_config.get("names", {})
    if isinstance(names, list):
        names = dict(enumerate(names))
    valid = True
    print(f"\n{modality}: {root}")
    for split in ("train", "val"):
        images = [p for p in (root / "images" / split).glob("*")
                  if p.is_file() and p.suffix.lower() in image_extensions]
        labels = list((root / "labels" / split).glob("*.txt"))
        missing_labels = {p.stem for p in images} - {p.stem for p in labels}
        orphan_labels = {p.stem for p in labels} - {p.stem for p in images}
        valid = valid and bool(images) and not missing_labels and not orphan_labels
        counts = Counter()
        negatives = 0
        for label in labels:
            rows = [row for row in label.read_text().splitlines() if row.strip()]
            negatives += not rows
            counts.update(int(row.split()[0]) for row in rows)
        print(split, {"images": len(images), "labels": len(labels),
                      "negative_frames": negatives, "missing_labels": len(missing_labels),
                      "orphan_labels": len(orphan_labels)})
        print({names.get(key, str(key)): count for key, count in sorted(counts.items())})
    if valid:
        ready_modalities.append(modality)
    else:
        print("Check the folder layout, nonempty splits, and image/label pairing before training.")

## 2. Train with the documented final settings

The two streams are optimized independently. Set `RUN_TRAINING = True` in the configuration cell and rerun it to enable training. A GPU is required by this example. RGB at 1280 px and batch 16 may exceed available GPU memory; reduce the batch size if necessary and record the change.


In [ ]:
if not RUN_TRAINING:
    print("Training skipped. Configure your own data, set RUN_TRAINING = True, and rerun the configuration cell.")
else:
    if set(ready_modalities) != set(MODALITIES):
        raise RuntimeError("Dataset setup is incomplete. Fix the messages above before enabling training.")
    if not torch.cuda.is_available():
        raise RuntimeError("Select a GPU runtime and rerun the setup/configuration cells before training.")
    for modality in MODALITIES:
        config = FINAL_RUNS[modality]
        model = YOLO("yolov8n.pt")  # Generic pretrained initialization, not the project-trained model.
        model.train(
            data=str(config["data"]),
            epochs=100,
            imgsz=config["imgsz"],
            batch=16,
            optimizer="SGD",
            patience=20,
            device=0,
            name=f"{modality}_yolov8n_{config['imgsz']}px_sgd",
            save=True,
            plots=True,
        )

## 3. Reported experiment evidence

The original saved runs reported Thermal mAP@50 = **84.70%** at 640 px and RGB mAP@50 = **85.93%** at 1280 px. These are historical validation results, not expected scores on your own data. For the experiment sweep, use `imgsz` values 640, 960, and 1280 and compare SGD with AdamW.

The calculation below only recomputes the published count ratio from saved totals; it does not run inference or re-evaluate annotations. The OOD result is a sequence-level proxy, not object-level recall.


In [ ]:
detections, annotations = 555, 572
coverage = detections / annotations
print(f'OOD count coverage: {coverage:.2%}')
print('Do not interpret this as object-level recall without one-to-one matching.')